# ZTE — the evidence suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/main/notebooks/tbme/zte_tbme.ipynb)

**Upload this notebook on its own.** It clones the repo, reads your Drive, and runs the eight experiments that
together decide what this programme is entitled to claim. It ends with one command that assembles every measured
number beside the brain-free floor it has to clear.

| § | Experiment | What it settles | Trains what | Roughly |
| --- | --- | --- | --- | --- |
| **6** | Granularity ablation | Does sentence, word or token alignment recover anything spelling does not? | 36 encoders | **~84 GPU-h** + 3 h of audits |
| **7** | Passage confound | Is cross-subject retrieval semantics, or memorised passages? | 3 encoders | ~5 GPU-h + 30 min |
| **8** | Decoder reality check | Does generated text carry the EEG, or the LM's prior? | 1 decoder over §7's encoder | ~2 GPU-h + 40 min |
| **9** | Anchor calibration | What does a new reader gain from a few labelled sentences, without retraining? | nothing — reads §6's ZAB arm | ~20 min |
| **10** | Semantic hard negatives | Does punishing length- and piece-matched confusions raise retrieval? | 1 encoder (§6's ZAB arm is the pair) | ~2 GPU-h |
| **11** | Architecture benchmark | Does the conformer beat EEGNet and DeepConvNet through the identical pipeline? | 2 encoders | ~4 GPU-h |
| **12** | Physiological interpretability | *When* in the word, and *where* on the scalp, does the readout come from? | nothing — reads §6's ZAB arm | ~15 min |
| **13** | Twelve-fold LOSO | The population number, mean ± sd, not one lucky holdout | nothing — reads §6 | minutes |
| **14** | **The evidence board** | Every claim, its floor, its verdict | nothing | seconds |

**Run it in order.** §9, §12 and §13 read checkpoints §6 trains, and §8 reads one §7 trains, so they are not
independent entry points — `resolve_ckpt` raises rather than guessing, so a skipped section fails loudly rather
than quietly. The one section that stands alone is **§6a**, the model-free piece oracle: it needs no checkpoint at
all and it is the number that decides whether the rest is worth the GPU time.

## This is a multi-day run. Two things decide whether day two works.

**Set `RESUME_DATE` in section 4 the moment you come back.** A session with `RESUME_DATE = None` opens a folder
named by *today's* date. On day two that is a new, empty directory: `--resume` looks for its checkpoints under it,
finds none, and retrains all thirty-six folds from scratch. Section 4 prints the date it opened — write it down, and
put it in `RESUME_DATE` before re-running anything.

**Everything is resumable, and nothing is idempotent by accident.** Training resumes from `last.pt`; the audits skip
themselves when their inputs are unchanged, through a `.zte-done` stamp that covers the checkpoint hash, the dataset
key and every flag. Re-running a finished cell costs the time to hash a checkpoint.

## How to read a number in this project

Three rules, and none of them is stylistic.

1. **`held_out_retrieval`, never `sentence_retrieval`.** The pooled number is computed over the training subjects
   too, so it rewards memorising the brains you have rather than reaching the one you do not. It inverted the
   champion once already: an arm with a pooled Top-1 of 0.043 scored 4 hits in 700 held out, and an identical re-run
   gave 2.
2. **Every retrieval number is read against a brain-free floor.** On the real 700-sentence gallery the word count
   alone retrieves **53** sentences and the total sub-word piece count **71**. The best encoder this programme has
   trained retrieves **33**. A number quoted without its floor is not evidence, whatever its *p*-value.
3. **Top-*k* is a hit count out of the queries actually scored, with an exact binomial tail — never a bare rate.**
   At chance 1/700, Top-1 expects exactly one hit, so a headline of "0.006 versus 0.001" is three hits and a coin.

Section 14 enforces all three mechanically: a row with no floor renders as `not measured`, and a row whose
confidence interval straddles its floor renders as below it. **Nothing on that board can go green by accident.**

## What is already known, before you run anything

So that no cell below comes as a surprise, and so no result here is oversold:

- The three alignment levels have been measured over twelve folds. **None clears the length-oracle floor**, and
  `token` — the arm most exposed to the spelling channel — is nominally highest. That ordering is the confound
  signature, not a win.
- The parallax transfer matrix already shows a code reaching a never-seen subject reading never-seen sentences at
  rank percentile ~0.95, with NR→SR the *strongest* cell. That is the passage-confound answer, and it is a positive
  result.
- The encoder supplies ~1.7–2.0 bits of sentence identity against the 9.45 a sentence needs. **Expect an honest null
  on generation**, and read section 8 as a control experiment rather than a headline.
- No trained checkpoint has an attentive temporal pool, so section 12 measures *occlusion* rather than attention.
  See its own preamble for why that is the better instrument anyway.

## 1 · Provision the runtime

Installs `uv`, clones or refreshes the repo, and builds the pinned Python 3.14 venv that every `!uv run` below uses.
The kernel you are typing in is Colab's own older interpreter and never imports `zte`.

In [ ]:
%%bash
pip install -q uv
# Work whether this is a fresh runtime (/content), a re-run already inside zte/, or a restored session.
if [ -f pyproject.toml ]; then :
elif [ -d zte/.git ]; then cd zte
else git clone --depth 1 https://github.com/victor-iyi/zte.git --branch main && cd zte
fi
git fetch --depth 1 origin main && git reset --hard FETCH_HEAD
echo "ZTE @ $(git rev-parse --short HEAD): $(git log -1 --pretty=%s)"
uv python install 3.14
uv sync --all-groups

## 2 · Wire the kernel

`colab()` is the only route into the package: it runs one `zte-colab` subcommand in the venv and returns the JSON it
printed. Nothing below computes with ZTE in this kernel — it renders payloads the venv produced.

In [ ]:
import json
import os
import platform
import subprocess
from typing import Any


def colab(command: str, *args: str) -> dict[str, Any]:
    """Runs one `zte-colab` subcommand in the provisioned venv and returns the JSON object it printed.

    This is the notebook's only route into ZTE. The package runs on 3.14 inside the uv venv; this kernel is
    Colab's own older interpreter, so it renders payloads rather than computing them.
    """
    argv = ['uv', 'run', 'zte-colab', command, *args]
    done = subprocess.run(argv, capture_output=True, text=True, check=False)
    if done.returncode != 0:
        raise RuntimeError(f'`{" ".join(argv)}` failed:\n{done.stderr[-3000:]}')

    return json.loads(done.stdout)


# Enter the repo in the notebook kernel, so relative paths and every subprocess resolve. A %%bash `cd` cannot
# do this: it dies with its own shell.
if os.path.isdir('zte') and not os.path.isfile('pyproject.toml'):
    os.chdir('zte')

ENV = colab('env')
os.environ.update(ENV['env'])

try:
    from google.colab import userdata  # type: ignore[import-untyped]

    _hf = userdata.get('HF_TOKEN')
except Exception as exc:  # not on Colab, or the secret is not granted to this notebook
    _hf, _ = None, print(f'HF_TOKEN unavailable ({type(exc).__name__}) - Hub downloads will be unauthenticated.')
if _hf:
    os.environ['HF_TOKEN'] = _hf
    print('HF_TOKEN loaded - authenticated HuggingFace Hub downloads enabled.')

print(f'repo   : {ENV["root"]}')
print(f'venv   : Python {ENV["venv"]["python"]} - zte {ENV["venv"]["zte"]}   <- every `!uv run` command')
print(f'kernel : Python {platform.python_version()}   <- this cell; renders payloads, never imports zte')

## 3 · What hardware did you get

Sections 6, 10, 11 and 13 train and need an accelerator. Everything else reads checkpoints and runs on CPU.

In [ ]:
plan, res = ENV['plan'], ENV['resources']

print(f'backend   : {plan["backend"]}  -  device {plan["device"]}  -  autocast {plan["autocast_dtype"]}')
print(f'resources : {res["ram_gb"]} GB RAM - {res["cpu_count"]} cores - {res["free_disk_gb"]} GB free')
print(f'gpu       : {res.get("gpu") or "none - the training sections need one"}')
if not res.get('gpu'):
    print('\nNo accelerator: Runtime -> Change runtime type -> GPU, or run only the read-only sections.')

## 4 · Drive is the workspace

Every artifact lands in a dated session folder on Drive, so a reclaimed VM costs you nothing. Set `RESUME_DATE` to
an existing folder name to continue a session; leave it `None` to start today's.

In [ ]:
from google.colab import drive  # type: ignore[import-untyped]

drive.mount('/gdrive')

In [ ]:
# SET THIS on day two and after. None opens a folder named by today's date -- a new, empty session, in which
# `--resume` finds no checkpoints and every fold retrains from scratch. The date this cell prints is the one to
# put here when you come back.
RESUME_DATE: str | None = None
# 'auto' writes runs to Drive when it is mounted, and to the local disk otherwise. On Colab that is what keeps a
# twelve-fold sweep inside the VM's disk: 54 runs are ~27 GB of checkpoints beside an 11-24 GB bundle.
WRITE_MODE: str = 'auto'
ZTE_DRIVE: str = '/gdrive/My Drive/Sharables/ZTE'

_resume = ('--resume-date', RESUME_DATE) if RESUME_DATE else ()
SESSION = colab('session', '--drive', ZTE_DRIVE, '--write-mode', WRITE_MODE, *_resume)
os.environ.update(SESSION['env'])

RUN_DATE: str = SESSION['run_date']
DATA_DIR: str = SESSION['data_dir']
DRIVE_DIR: str = SESSION['session_dir']
DRIVE_ANALYSIS: str = SESSION['drive_analysis']
LOCAL_RUNS: str = SESSION['local_runs']
OUT_ROOT: str = SESSION['out_root']
DRIVE_BACKUP: str = SESSION['drive_backup']
PREPARED_LOCAL: str = SESSION['prepared_local']
PREPARED_DRIVE: str = SESSION['prepared_drive']

# Every training cell below passes these, and names its run explicitly. The `_s42` in a run directory comes from
# that literal name, not from `--seed`, which only pins `train.seed`; `--data-cache` and `--drive-backup` are what
# let a reclaimed VM resume; `--spatial exact` builds the montage CSV the configs name but do not create.
SEED: int = 42
# The single-holdout subject for every arm that is not a twelve-fold sweep. Sections 6c and 9-12 all read a
# checkpoint named with it, so it is fixed once here rather than per section.
HOLDOUT: str = 'ZAB'
TRAIN_FLAGS: str = (
    f'--seed {SEED} --data-cache "{PREPARED_LOCAL}" --data-cache-remote "{PREPARED_DRIVE}" '
    f'--drive-backup "{DRIVE_BACKUP}" --spatial exact'
)

# Everything this notebook measures lands under one root, which is also the only thing section 14 has to read.
# `DRIVE_ANALYSIS` is a path under the Drive session whether or not Drive is actually mounted, so without the mount
# it would name a directory nothing can write to. Fall back to the local tree, as the other notebooks' `durable()`
# helper does -- the audits then survive the cell, and only the Drive copy is lost.
SUITE: str = f'{DRIVE_ANALYSIS}/evidence_suite' if SESSION['drive_mounted'] else 'res/evidence_suite'

# A new session on day two is the one mistake that silently costs a whole sweep: `--resume` looks under
# `OUT_ROOT`, which is this session's folder, so an earlier session's checkpoints are invisible and every fold
# retrains. Check for one and say so loudly rather than trusting the reader to remember.
SESSIONS = colab('runs', '--drive', ZTE_DRIVE)['sessions']
EARLIER = [path for path in SESSIONS if RUN_DATE not in path]

if EARLIER and not SESSION['resumed']:
    print('=' * 100)
    print('STOP. This is a NEW session, and earlier ones on Drive already hold runs:')
    for path in EARLIER[:6]:
        runs = colab('runs', '--drive', '', '--experiments', path)['runs']
        arms = sum(1 for run in runs if run['name'].startswith(('align_', 'parallax_', 'benchmark_', 'decode_')))
        if arms:
            print(f'    {path.split("/")[-2]}  ->  {arms} run(s) from this campaign')
    print()
    print('Set RESUME_DATE above to that date and re-run this cell. Continuing as-is writes into an empty')
    print('directory, so --resume finds nothing and every fold retrains from scratch.')
    print('=' * 100)
    print()

print(f'session   : {RUN_DATE}   ({"resumed" if SESSION["resumed"] else "new"})')
print(f'raw data  : {DATA_DIR}   (present: {SESSION["data_dir_present"]})')
print(f'runs ->   : {OUT_ROOT}')
print(f'suite ->  : {SUITE}')
if not SESSION['drive_mounted']:
    print('\nDrive is not mounted: runs and audits stay on this machine and nothing is mirrored.')
if not SESSION['data_dir_present']:
    print('\nZuCo is not at that path. Sections that read the corpus will fail until it is.')

### 4a · Helpers this notebook uses everywhere

`audit()` and `show()` are the pair every experiment below ends with: one reads an audit's JSON through the bridge,
the other displays the Markdown that audit's own CLI wrote. Nothing is recomputed in this kernel, so what you read
is what the run recorded.

In [ ]:
import pathlib


def resolve_ckpt(run_name: str, which: str = 'best') -> str:
    """Finds a run's checkpoint, Drive first, so a fresh VM can audit a session it did not train.

    A missing `best.pt` never falls back to `last.pt`: they are different models, and swapping them silently
    misattributes the number.
    """
    for run in colab('runs', '--drive', ZTE_DRIVE, '--experiments', LOCAL_RUNS, '--run', run_name)['runs']:
        if path := run['checkpoints'][which]:
            print(f'{which}.pt for {run_name}: {"Drive" if run["source"] == "drive" else "local disk"}\n  {path}')
            return path

    raise FileNotFoundError(f'no {which}.pt for {run_name!r} on Drive or locally; train it first.')


def audit(kind: str, directory: str, markdown: bool = True) -> dict[str, Any]:
    """Reads one audit's JSON (and its Markdown) out of a directory, recomputing nothing."""
    flags = [] if markdown else ['--no-markdown']
    payload = colab('audit', '--from', directory, '--kind', kind, *flags)['audits'][kind]
    if not payload['found']:
        print(f'no {kind} artifact at {payload["json_path"]} - run the cell above it first.')

    return payload


def show(payload: dict[str, Any]) -> None:
    """Renders an audit's own Markdown inline, so the notebook shows the report the CLI wrote."""
    from IPython.display import Markdown, display

    if payload.get('markdown'):
        display(Markdown(payload['markdown']))
    else:
        print('no rendered Markdown beside that artifact.')


def mirror_to_drive(sub: str = 'experiments') -> None:
    """Copy the VM's runs to Drive, minus what is rebuildable, so the session survives the machine."""
    where = ['--drive', ZTE_DRIVE, '--write-mode', WRITE_MODE, '--direction', 'up']
    payload = colab('mirror', *where, '--date', RUN_DATE, '--sub', sub)
    reason = payload['skipped_reason']
    if reason and 'same directory' in reason:
        return  # write_mode resolved to `drive`, so the runs are already there and a mirror is a copy onto itself.

    if reason:
        print(f'nothing mirrored: {reason}')
    else:
        print(f'{payload["src"]} -> {payload["dst"]}   ({payload["copied"]} copied, {payload["failed"]} failed)')


def num(value: Any, digits: int = 4) -> float | None:
    """Rounds a number for a table, passing None through.

    Every audit in this project reports an uncomputable cell as None rather than raising, so a rendering cell that
    called round() on one would take the whole table down over a single bad arm.
    """
    return round(float(value), digits) if isinstance(value, (int, float)) else None


def show_resources() -> None:
    """Prints RAM / GPU / disk as they stand now, so an out-of-memory kill is predictable rather than a mystery."""
    r = colab('env')['resources']
    gpu = f'{r["gpu"]["name"]} ({r["gpu"]["total_gb"]} GB)' if r['gpu'] else 'none'
    print(f'RAM {r["ram_gb"]} GB - {r["cpu_count"]} cores - {r["free_disk_gb"]} GB free disk - GPU {gpu}')


show_resources()

In [ ]:
# Rendering only: these read the JSON that `zte-colab` and the audit CLIs already produced.
import pandas as pd
import plotly.graph_objects as go

### 4b · What re-running costs, command by command

Every cell below is safe to re-run, and **no training, audit, calibration or occlusion pass is ever repeated**.
Measured on a full synthetic replay of this notebook's own commands: a cold pass writes 55 artifacts, the second
rewrites 4, the third rewrites 2 — the two being `zte-loso-summary`, which is a pure reader over a second of work.
Nothing that costs GPU time runs twice.

Two different mechanisms do that, and it is worth knowing which is which.

| command | how it skips | what invalidates it |
| --- | --- | --- |
| `zte-prepare` | bundle key over the dataset config | a different representation, window, task set or subject list |
| `zte-run` | `--resume`: a finished run returns in under a second | a new run name, or a stage whose inputs moved |
| `zte-audit` · `zte-rebaseline` · `zte-decode` · `zte-calibrate` · `zte-parallax transfer` · `zte-lens` | a `.zte-done` stamp beside the first artifact | the checkpoint's SHA-256, the dataset key, or **any** flag |
| `zte-levels` · `zte-evidence` | a `.zte-done` stamp that also fingerprints the artifacts read | a new fold, a re-run audit, a changed file size |
| `zte-parallax report` · `zte-loso-summary` | nothing — they re-read and rewrite each time, in about a second | — |

**`--force` is deliberately absent everywhere below.** Passing it would defeat the stamp and repeat the work the
stamp exists to skip. The two readers that aggregate — `zte-levels` and `zte-evidence` — hash the *contents* of what
they read, so a new fold or a re-run audit rebuilds them on its own; nothing has to be forced.

Where things land, and why none of it is lost with the VM:

| what | where | durability |
| --- | --- | --- |
| checkpoints, per-run evaluation | `OUT_ROOT` | Drive directly when mounted; `--drive-backup` mirrors per epoch otherwise |
| the prepared bundle | `PREPARED_LOCAL`, layered over `PREPARED_DRIVE` | built once, staged down on any later VM |
| the montage CSV | `res/montage_gsn105.csv`, published to the bundle store | rebuilt only if the store loses it |
| every audit and the board | `SUITE` | under the Drive session, or `res/` when Drive is absent |

In [ ]:
# One metadata check per dataset, no building. Safe to run on every session, including over a Drive mount.
!uv run zte-prepare --check --root "{DATA_DIR}" --configs experiments/alignment experiments/parallax experiments/benchmark --cache-dir "{PREPARED_LOCAL}" --cache-remote "{PREPARED_DRIVE}"

## 5 · Prepare the data once, on Drive, and never again

The prepared bundle is keyed by a full-config hash and staged on the roomiest volume the VM has. Re-running this is
a no-op once the bundle exists, and it is mirrored to Drive so the next session skips it entirely.

This is the only cell that touches the 17–23 GB raw archives.

In [ ]:
!uv run zte-prepare --root "{DATA_DIR}" --configs experiments/alignment experiments/parallax experiments/benchmark --cache-dir "{PREPARED_LOCAL}" --cache-remote "{PREPARED_DRIVE}"

## 6 · Experiment 1 — the granularity ablation

One contrastive term, moved between three units and nothing else changed:

| level | the unit it aligns | frozen target |
| --- | --- | --- |
| `sentence` | the pooled sentence vector | a frozen E5 sentence embedding |
| `word` | one fixated word = one EEG token | a frozen word vector |
| `token` | four fixed intra-word slices of one word | the LM's sub-word embeddings |

The claim under test is that finer alignment recovers more. The claim that has to be *excluded first* is that finer
alignment recovers more **spelling**.

### 6a · The floor, with no model involved

Run this before anything else. Four brain-free signatures, each scored as a retrieval oracle over your own
700-sentence gallery — no checkpoint, no training, nothing but the reference text.

| signature | what it is told, and nothing else |
| --- | --- |
| `words` | the word count |
| `total` | the total sub-word piece count, one integer per sentence |
| `multiset` | the piece counts with their order destroyed |
| `profile` | the ordered per-word piece counts |

`information_bits` is $\log_2 n - \frac{1}{n}\sum_i \log_2 m_i$, where $m_i$ counts the gallery sentences sharing
sentence *i*'s signature. On 700 sentences the ceiling is $\log_2 700 = 9.4512$ bits.

Watch **`alignment_coverage`** — the fraction of ZuCo words that matched their own reference text. Below about 0.99
the piece counts are partly wrong and the bits are not trustworthy.

In [ ]:
# `--config` decides which dataset is built -- representation, window, tasks -- and so which bundle key is looked
# up. With `--root` alone the defaults build a band-power/128-sample dataset that keys nowhere near the prepared
# raw bundle and re-parses every archive over the Drive mount.
!uv run zte-audit --config experiments/alignment/sentence/combined.yaml --root "{DATA_DIR}" \
    --piece-oracle --out "{SUITE}/audit/confound_audit.md"

In [ ]:
_audit = audit('oracle', f'{SUITE}/audit', markdown=False)['payload']
if not _audit:
    raise SystemExit('No confound audit yet -- run the cell above before this one.')

ORACLE = _audit['piece_oracle']

rows = [
    {
        'signature': name,
        'Top-1': round(block['top1'], 4),
        'hits/700': round(block['top1'] * block['n']),
        'bits of 9.4512': round(block['information_bits'], 3),
        'unique fraction': round(block['unique_fraction'], 3),
    }
    for name, block in ORACLE['oracles'].items()
]
display(pd.DataFrame(rows))

print(f'tokeniser : {ORACLE["tokenizer"]}')
print(
    f'coverage  : {ORACLE["alignment_coverage"]:.5f}'
    f'{"" if ORACLE["alignment_coverage"] > 0.99 else "   <-- below 0.99, the piece counts are partly wrong"}'
)
print(f'gate      : {ORACLE["gate_signature"]} at Top-1 {ORACLE["gate_top1"]:.4f} ({ORACLE["gate_bits"]:.2f} bits)')
print(f'ceiling   : {ORACLE["ceiling_signature"]} at Top-1 {ORACLE["ceiling_top1"]:.4f}')

**How to read that.** `gate` is the floor a *fixed* sub-token count can actually reach — what this repository builds,
because `objective.token_sub_tokens` is a constant per word and never the count of pieces the reference spells that
word in. `ceiling` is what a design that sized a word's EEG by its own piece count would have handed over.

The number to compare either against is the best held-out Top-1 this programme has measured. Section 6c prints it
beside the floor rather than leaving you to do the arithmetic.

### 6b · Train the three levels — optional, and expensive

Twelve folds × three levels is roughly 110 GPU-hours. **If the alignment arms already exist on Drive, skip this and
go straight to 6c**, which reads them. The planner below says what is already done, so a reclaimed VM resumes
exactly where it stopped.

In [ ]:
LEVELS = ['sentence', 'word', 'token']

# `--regimes combined` matters: the planner's `parallax` regime means the per-task arms of each alignment level
# (align_sentence_nr and friends), which this notebook does not train. Leaving it in would report them as pending
# for ever. Section 7's per-task encoders are a different family, under experiments/parallax/.
# `--tiers` and `--regimes` both default to the whole 51-run campaign, which includes the per-task alignment arms
# and the seed-43/44 reseeds that no cell here trains -- so the defaults would report hours outstanding for ever.
PLAN = colab(
    'sweep',
    'status',
    '--tiers',
    'power',
    '--levels',
    *LEVELS,
    '--regimes',
    'combined',
    '--out-root',
    OUT_ROOT,
    '--drive',
    ZTE_DRIVE,
)

for block in PLAN['progress']['tiers']:
    print(f'{block["tier"]:<11} {block["done"]:>3}/{block["total"]:<3} done  -  {block["hours_remaining"]:.1f} h left')

nxt = PLAN['next']
print()
print(f'next up: {nxt["run_name"]}  ->  {nxt["out_dir"]}' if nxt else 'nothing left to train at these levels.')

Each fold is one `zte-run` with `--resume`, which is idempotent and skips finished work. `FOLDS` lists all twelve
ZuCo subjects; trim it to shorten the sweep, but **a single holdout is not a population result** and section 13 is
where the mean ± sd comes from.

In [ ]:
FOLDS = ['ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH']

# Fold OUTER, level INNER. A reclaimed VM is certain across this many hours, and this ordering means whatever
# prefix survives is a matched comparison across all three levels rather than one level finished and two at zero.
for holdout in FOLDS:
    for level in LEVELS:
        cfg = f'experiments/alignment/{level}/combined.yaml'
        name = f'align_{level}_combined_lo{holdout}_s{SEED}'
        print(f'\n===== {name} =====')
        !uv run zte-run --config "{cfg}" --root "{DATA_DIR}" --name "{name}" --loso-holdout {holdout} \
            --out-root "{OUT_ROOT}" {TRAIN_FLAGS} --resume
    mirror_to_drive()

### 6c · Give every fold its floor

**Do not skip this.** `zte-levels` reads each run's brain-free floors from `<run>/rebaseline/rebaseline.json`, which
is where `zte-rebaseline` writes by default. Without it the cross-level table renders every level as
*floor not measured* -- the numbers are there but unread, and the section has no verdict.

Section 6a scored the piece oracle over the corpus alone, so it had no encoder to judge and `observed_top1` was
null. Against a checkpoint it becomes a verdict.

About five minutes per run, so roughly three hours for all thirty-six -- 3% of what training them cost. Each is
guarded by its own `.zte-done` stamp, so a reclaimed VM resumes rather than repeating.

In [ ]:
for holdout in FOLDS:
    for level in LEVELS:
        run = f'align_{level}_combined_lo{holdout}_s{SEED}'
        ckpt = resolve_ckpt(run)
        # No --out: the default is `<run>/rebaseline`, which is exactly where zte-levels looks for the floor.
        !uv run zte-rebaseline --ckpt "{ckpt}" --root "{DATA_DIR}" --piece-oracle
mirror_to_drive()

### 6d · The cross-level table, and the floor every level is read against

`zte-levels` reads the runs that already exist — it loads no model and re-scores no query — groups them by level,
aggregates across folds with a **sample** (n−1) standard deviation, and prints each level against its floor.

Two galleries are shown deliberately. The unstratified one is what a naive paper reports; the length-stratified one
is the honest cell, because on the unstratified gallery word count alone can answer the query.

In [ ]:
# `_combined_` is load-bearing. A bare `align_*` would also sweep in section 10's `align_sentence_hardneg` arm,
# which `zte-levels` classifies as a sentence run and would average into the sentence row -- a different objective
# silently mixed into the granularity comparison.
!uv run zte-levels --root "{OUT_ROOT}" --pattern "align_*_combined_lo*" --out "{SUITE}/levels"

In [ ]:
show(audit('levels', f'{SUITE}/levels'))

In [ ]:
LEVELS_PAYLOAD = audit('levels', f'{SUITE}/levels', markdown=False)['payload']

fig = go.Figure()
names = [block['level'] for block in LEVELS_PAYLOAD['levels']]
strat = [(block.get('length_stratified') or {}) for block in LEVELS_PAYLOAD['levels']]
floors = [(block.get('length_floor') or {}).get('rank_percentile') for block in LEVELS_PAYLOAD['levels']]

fig.add_bar(
    x=names,
    y=[cell.get('rank_percentile') for cell in strat],
    error_y={'type': 'data', 'array': [cell.get('rank_percentile_sd') or 0 for cell in strat]},
    name='encoder (length-stratified)',
)
if any(floors):
    fig.add_hline(
        y=max(f for f in floors if f),
        line_dash='dash',
        line_color='crimson',
        annotation_text='length oracle (tol=1) - the brain-free floor',
        annotation_position='top left',
    )
# Derived, not fixed: a hard lower bound renders any bar beneath it as blank, and a bar autoscaled from 0 makes
# 0.92 and the 0.9525 floor indistinguishable. Frame the region the comparison actually lives in.
drawn = [v for v in [cell.get('rank_percentile') for cell in strat] + floors if isinstance(v, (int, float))]
low = min(drawn) - 0.02 if drawn else 0.85

fig.update_layout(
    title='Held-out rank percentile by alignment level, against the floor it must clear',
    yaxis_title='rank percentile',
    yaxis_range=[low, 1.0],
    height=420,
    showlegend=True,
)
fig.show()

**What this measures.** If every bar sits below the dashed line, the granularity comparison has no subject: the
levels are being ranked against each other inside a region a single integer already dominates. That is the measured
outcome on real ZuCo as of 2026-08-24, and `zte-levels` prints the confound-signature sentence rather than naming a
winner. A null is a finding, and this one is a *limit*: it says sub-word structure is not recoverable from
non-invasive EEG on this corpus, because any apparent recovery is bounded above by the spelling.

## 7 · Experiment 2 — the passage confound

The strongest objection to any cross-subject retrieval number is that the model memorised the passage rather than
read the meaning. ZuCo makes that objection testable, because the tasks' sentence sets are **disjoint**: no sentence
appears under both normal reading (NR) and sentiment reading (SR).

So train on one task and evaluate on another. An off-diagonal cell faces a **never-seen subject reading never-seen
stimuli**. If retrieval survives that, it is not passage memorisation.

`stimulus_novelty` measures the overlap rather than assuming it: a cell with any shared stimulus is flagged
`novel_stimuli: false` and cannot carry the claim.

In [ ]:
TASKS = ['NR', 'SR', 'TSR']

for task in TASKS:
    cfg = f'experiments/parallax/parallax_{task.lower()}.yaml'
    name = f'parallax_{task.lower()}_lo{HOLDOUT}_s{SEED}'
    print(f'\n===== {name} =====')
    !uv run zte-run --config "{cfg}" --root "{DATA_DIR}" --name "{name}" --loso-holdout {HOLDOUT} \
        --out-root "{OUT_ROOT}" {TRAIN_FLAGS} --resume
    mirror_to_drive()

In [ ]:
for train_task in TASKS:
    ckpt = resolve_ckpt(f'parallax_{train_task.lower()}_lo{HOLDOUT}_s{SEED}')
    for eval_task in TASKS:
        print(f'\n===== {train_task} -> {eval_task} =====')
        # `--seed` here names the cell directory (`<train>_to_<eval>_s<seed>`) as well as seeding the bootstrap,
        # and it defaults to 0 -- which would label cells scored from s42 checkpoints as seed 0.
        !uv run zte-parallax transfer --ckpt "{ckpt}" --root "{DATA_DIR}" --eval-task {eval_task} \
            --holdout {HOLDOUT} --seed {SEED} --out "{SUITE}/transfer"

In [ ]:
!uv run zte-parallax report --transfers "{SUITE}/transfer" --out "{SUITE}/transfer"
show(audit('transfer', f'{SUITE}/transfer'))

**How to read the matrix.** Chance rank percentile is 0.5, so a confidence interval bracketing 0.5 is a null — and a
null here is a finding: it says the task-specific code did not transfer.

On the runs measured 2026-08-24, **NR→SR is the strongest cell in the matrix** at rank percentile 0.9595, above the
NR→NR diagonal at 0.9488, with 5 of 400 Top-1 hits at *p* = 0.0036. A model trained on normal reading reads
sentiment-reading sentences it has never seen, in a brain it has never seen, at least as well as it reads its own
task. **That is evidence against passage memorisation**, and it is the most positive result in this notebook.

Two qualifiers travel with it. Rank percentile resists the length confound; the Top-*k* on that cell does not, and 5
hits is below what settles a Top-*k* comparison. And TSR carries no measurable in-task content signal at all.

## 8 · Experiment 3 — the decoder reality check

This section is a **control experiment, not a headline.** Read it as the argument for why retrieval is the only
sound readout right now, rather than as a decoding result.

The arithmetic that makes it a control: the encoder supplies ~1.7–2.0 bits of sentence identity, a 19.6-word
sentence needs ~190, so free generation has about **1%** of what it requires. An honest null is the expected
outcome.

`zte-decode` runs the headline decode and then every pre-registered brain-independent control through the
*identical* `generate_from_prefix` path, so nothing differs between them but the conditioning:

| control | what it replaces | what it isolates |
| --- | --- | --- |
| `mean_prefix` | one prefix for every reading | anything the LM says regardless of input |
| `null_prefix` | no prefix at all | the LM's unconditional prior |
| `phase` | phase-scrambled EEG, power spectrum preserved | spectral shape without time structure |
| `noise` | noise-matched EEG at the **encoder input** | the encoder's response to matched noise |
| `noise_prefix` | noise-matched **z** straight into the bridge | how much of the readout is the LM's prior |
| `shuffled_z` | another reading's prefix | "any well-formed prefix" |
| `length_only` | a prefix built from word count alone | the 5.14-bit length channel |
| `mismatch` | a deliberately wrong pairing | the scoring itself |

`noise_prefix` is the one that answers the question directly. It is drawn with matched per-feature mean and variance
against the real **z** cloud — an off-manifold standard normal would be a trivially weak control and would let the
decoder look good for the wrong reason.

**The verdict gate ANDs over these**, and an unavailable or skipped control **fails** its clause. Adding a control
therefore tightens the gate, which is the intent.

In [ ]:
# The decoder trains over a FROZEN encoder, so it needs that checkpoint resolved at run time. Every decoder YAML
# carries an `encoder_ckpt` pointing at a local path from the session that wrote it, which a fresh VM does not
# have -- `--encoder-ckpt` overrides it, and section 7 has already trained the encoder this arm reads.
DECODE_CFG = 'experiments/decoder/decode_parallax_nr.yaml'
DECODE_RUN = f'decode_parallax_nr_lo{HOLDOUT}_s{SEED}'
DECODE_OUT = f'{SUITE}/decode'

ENCODER_CKPT = resolve_ckpt(f'parallax_nr_lo{HOLDOUT}_s{SEED}')

In [ ]:
# `--loso-holdout` is deliberately absent: this is a decoder arm, its config already names
# `train.loso_holdout_subject` inside an honest `by_subject_and_stimulus` split, and the flag would force
# `by_subject_loso`, which shares every stimulus between train and val. `zte-run` refuses it here.
# `--skip-eval`: this config sets eval_generation / eval_rescoring / eval_capacity, so the run would decode the
# whole split into its own directory and the cell below would then repeat it into the suite. One pass, landing
# where the evidence board reads.
!uv run zte-run --config "{DECODE_CFG}" --root "{DATA_DIR}" --name "{DECODE_RUN}" \
    --encoder-ckpt "{ENCODER_CKPT}" --out-root "{OUT_ROOT}" {TRAIN_FLAGS} --skip-eval --resume
mirror_to_drive()

In [ ]:
DECODE_CKPT = resolve_ckpt(DECODE_RUN)
!uv run zte-decode --ckpt "{DECODE_CKPT}" --root "{DATA_DIR}" --split test --capacity --out "{DECODE_OUT}"

In [ ]:
READINGS = colab('readings', '--from', DECODE_OUT, '--rows', '8')

verdict = READINGS['verdict']
print(f'generation_above_controls : {verdict["above_controls"]}')
print(f'worst control             : {verdict.get("worst_control")}  at  {verdict.get("worst_ci")}')
print(f'permutation p             : {verdict.get("permutation_p")}')
print(f'prefix-influence KL       : {verdict.get("prefix_kl")}  (floor {verdict.get("min_prefix_kl")})')
absent = verdict.get('controls_absent') or []
print(f'controls absent (= failing): {", ".join(absent) if absent else "none"}')
print()
for name, passed in (verdict.get('clauses') or {}).items():
    print(f'  {"PASS" if passed else "FAIL"}  {name}')

In [ ]:
rows = READINGS.get('readings') or []
if not rows:
    print(f'No scoreable readings in {DECODE_OUT}: {READINGS.get("reason") or "the decode produced none"}.')
else:
    frame = pd.DataFrame(
        [
            {'condition': cond['name'], **{k: num(v) for k, v in (cond['scores'] or {}).items()}}
            for cond in rows[0]['conditions']
        ]
    )
    display(frame)

    print('\nOne reading, every condition. If the noise_prefix row scores like the hypothesis row on content')
    print('metrics, the decode is the language model, not the brain.')

### The readout this project can prove

`--capacity` above swept the K-way menu as well, and that is the number worth carrying out of this section. A menu
asks a strictly easier question than free generation: given the held-out reading and *K* candidate sentences, does
the decoder score the one actually read above every distractor? Chance is exactly $1/K$, ties lose, and the pool is
length- and task-matched so counting words cannot win it.

Seven clauses must hold at *K* **and** at every smaller size swept: an honest `by_subject_and_stimulus`/`test`
split, a certifiable (never `open`) pool, a bootstrap CI lower bound above $1/K$, paired wins over `length_only`,
`shuffled_eeg` and `mismatch` on both a bootstrap CI and an exact sign test, and a permutation *p* below alpha.
`certified_k` is `None` when nothing certifies, and the failing clauses are named rather than hidden.

In [ ]:
CAPACITY = colab('capacity', '--from', DECODE_OUT)

sel = CAPACITY['selected']
print(
    f'score / pool  : {sel["score"]} / {sel["flavor"]}'
    f'{"   (substituted -- the run never swept what was asked for)" if sel.get("substituted") else ""}'
)
print(f'certified K   : {CAPACITY["certified_k"] or "nothing certified"}')
print(f'honest split  : {CAPACITY["honest_split"]}  ({CAPACITY.get("split_cell")})')
print()
for clause, ok in (CAPACITY['clauses'] or {}).items():
    print(f'  {"PASS" if ok else "FAIL"}  {clause}')

if CAPACITY.get('per_k'):
    display(pd.DataFrame(CAPACITY['per_k']))

**What a null here establishes.** If the EEG-conditioned decode does not beat `noise_prefix` on content metrics,
then content-metric scores of generated text are not a measure of neural decoding — they are a measure of the
language model's prior plus whatever length leaks through. That is a methodological claim about the field's
evaluation practice, and it is worth more than a weak positive.

It is also exactly why this project reports the powered readout as **decoder-rescoring retrieval over the
700-sentence gallery, and never as generation**. `verdict['generation_above_controls']` is the only gate that would
license a generation headline, and it must be `True` on an honest split with every control beaten.

## 9 · Experiment 4 — the anchor-calibration curve

**The deployment question.** A new reader arrives. They read *N* sentences whose text is known. With **no
retraining** — the encoder stays frozen, only a per-subject map into the shared space is fitted — how much does
their retrieval improve, and does it improve at all?

Three arms are scored at every anchor count, on one identical gallery:

| arm | what it is | what it controls for |
| --- | --- | --- |
| `uncalibrated` | frozen embeddings, anchors removed | the gallery got smaller, so retrieval got easier for free |
| `calibrated` | the map fitted on true (reading, text) pairs | the measurement |
| `shuffled` | the same map fitted on a **derangement** of those pairs | the transform's raw capacity |

**The `shuffled` arm is the one that makes this believable.** It fits the same map with the same parameter count on
the same number of pairs, with every anchor paired to the *wrong* reference. Whatever it lifts is what the transform
buys by existing rather than by being calibrated. A curve that does not beat its own shuffled control is not a
calibration result.

Two families bracket what any affine calibration could buy: `procrustes` is rotation-only, so it cannot inflate a
number by rescaling; `ridge` is strictly more expressive.

**Every anchor stimulus is removed from the query set *and* the gallery.** Leaving one in the gallery would let the
map place a sentence it was fitted on next to itself and manufacture the whole effect. That is why anchor count 0 is
re-scored on each reduced gallery rather than once on the full one — the columns you compare are the same problem.

Read `docs/CALIBRATION.md` for the algebra. Two things to know before the curve appears:

- The often-quoted "+0.0628 lift from 12 shared words" is a **cohesion** diagnostic at *word* level whose fitted map
  was never applied to a scored embedding. It is not evidence that retrieval moves.
- `dataset.raw_align_fit` defaults to `'all'`, so unlabelled per-subject whitening has **already** happened on the
  holdout. This curve measures what *labelled* anchors add on top of that.

In [ ]:
CALIB_CKPT = resolve_ckpt(f'align_sentence_combined_lo{HOLDOUT}_s{SEED}')
CALIB_OUT = f'{SUITE}/calibration'

!uv run zte-calibrate --ckpt "{CALIB_CKPT}" --root "{DATA_DIR}" --anchor-counts 0,10,25,50,100,200 --draws 5 --family both --out "{CALIB_OUT}"

In [ ]:
show(audit('calibration', CALIB_OUT))

In [ ]:
CALIB = audit('calibration', CALIB_OUT, markdown=False)['payload']
GALLERY = CALIB.get('headline_gallery', 'length_stratified')

fig = go.Figure()
for family, galleries in (CALIB.get('series') or {}).items():
    series = galleries.get(GALLERY) or {}
    x = series.get('anchor_counts') or []
    for key, label, dash in (
        ('calibrated_rank_percentile', f'{family} - calibrated', 'solid'),
        ('shuffled_rank_percentile', f'{family} - shuffled anchors (control)', 'dot'),
    ):
        if series.get(key):
            fig.add_scatter(x=x, y=series[key], mode='lines+markers', name=label, line={'dash': dash})
    if series.get('uncalibrated_rank_percentile'):
        fig.add_scatter(
            x=x,
            y=series['uncalibrated_rank_percentile'],
            mode='lines',
            name=f'{family} - uncalibrated, same gallery',
            line={'dash': 'dash'},
        )

fig.update_layout(
    title=f'Held-out rank percentile vs. calibration anchors ({GALLERY} gallery)',
    xaxis_title='labelled sentences from the new reader',
    yaxis_title='rank percentile',
    height=460,
)
fig.show()

for family, block in (CALIB.get('verdict') or {}).items():
    print(
        f'{family:<11} lift {block.get("lift")}  vs shuffled {block.get("shuffled_lift")}  '
        f'->  helps={block.get("helps")}  beats_shuffled={block.get("beats_shuffled")}'
    )
    print(f'            {block.get("verdict")}')

**How to read the curve.** The gap that matters is **calibrated minus shuffled**, not calibrated minus zero. The
report gives it as a *paired* per-draw difference with an interval (`margin_over_shuffled_ci`), because which
particular 10 sentences a reader happened to get is a real source of variance and every point is repeated over
seeded draws.

Three fields decide whether a point may be quoted at all, and the rendered report prints all three:
`underdetermined` (fewer anchors than dimensions — a `ridge` map at *N* = 10 in 768 dimensions is interpolating),
`degraded_fits` (draws that failed to fit; the fitter returns `None` and logs rather than silently becoming an
identity), and `saturated` (the reader had fewer stimuli than the count requested).

If this curve rises and beats its shuffled control, ZTE has a rapid-deployment property that needs no retraining —
the single most useful thing a BCI can have. If it does not, that is a deployment limit worth stating plainly.

## 10 · Experiment 5 — length- and piece-matched semantic hard negatives

The idea: punish the encoder for the *lazy semantic mistake*. If the target is "the dog bit the man", the negative
should be "the man bit the dog" — same length, same piece profile, same vocabulary, different meaning. Separate
those and the win cannot be surface form.

The repository already mined negatives by `surface_overlap − semantic_cosine`. That is the right ranking and the
wrong candidate set: it imposed **no length constraint**, so on this corpus a mined negative can still be told apart
by counting words, which teaches the encoder nothing. Since word count carries 5.14 of the 9.45 bits, matching on
length and on the sub-word piece budget is what makes a hard negative actually hard.

`hard_negative_strategy` selects the candidate set, and `hard_negative_in_loss` decides whether the mined table
narrows the **loss denominator** or only the batch composition. Both default off, so every existing run stays
byte-identical.

The pair below flips exactly those knobs against the measured sentence arm — one lever, so the difference is
attributable.

In [ ]:
HARDNEG_ARMS = {
    'align_sentence_combined': 'experiments/alignment/sentence/combined.yaml',
    'align_sentence_hardneg': 'experiments/alignment/sentence/hardneg.yaml',
}
HARDNEG_RUNS = [f'{stem}_lo{HOLDOUT}_s{SEED}' for stem in HARDNEG_ARMS]

for stem, cfg in HARDNEG_ARMS.items():
    name = f'{stem}_lo{HOLDOUT}_s{SEED}'
    print(f'\n===== {name} =====')
    !uv run zte-run --config "{cfg}" --root "{DATA_DIR}" --name "{name}" --loso-holdout {HOLDOUT} \
        --out-root "{OUT_ROOT}" {TRAIN_FLAGS} --resume
    mirror_to_drive()

In [ ]:
for run in HARDNEG_RUNS:
    ckpt = resolve_ckpt(run)
    !uv run zte-rebaseline --ckpt "{ckpt}" --root "{DATA_DIR}" --piece-oracle --out "{SUITE}/hardneg/{run}"

In [ ]:
rows = []
for run in HARDNEG_RUNS:
    payload = audit('rebaseline', f'{SUITE}/hardneg/{run}', markdown=False)['payload']
    if not payload:
        rows.append({'arm': run, 'rank pct': None, 'clears': 'NOT SCORED -- rerun its rebaseline cell'})
        continue
    honest = ((payload.get('grid') or {}).get('train_fitted') or {}).get('length_stratified') or {}
    floor = payload.get('floor_comparison') or {}
    hits = honest.get('top1')
    row = {'arm': run, 'rank pct': num(honest.get('rank_percentile'))}
    row['CI'] = [num(v) for v in (honest.get('rank_percentile_ci') or [])[1:]] or None
    row['Top-1 hits'] = None if hits is None else round(hits * honest['n_queries'])
    row['of'] = honest.get('n_queries')
    row['p'] = None if honest.get('top1_p') is None else f'{honest["top1_p"]:.3g}'
    row['length floor'] = num(floor.get('oracle'))
    row['clears'] = floor.get('clears_floor')
    row['gallery'] = floor.get('gallery')
    row['postproc'] = (payload.get('menu') or {}).get('postprocess_fit')
    rows.append(row)

display(pd.DataFrame(rows))

**What would count as a win.** Not a higher Top-1 on the unstratified gallery — that is where length lives. A win is
a higher **length-stratified rank percentile** whose interval clears the floor, on a matched pair that differs only
in the negative-mining strategy. Anything less is a different number, not a better model.

## 11 · Experiment 6 — the architecture benchmark

An ablation against a band-power MLP does not answer "is the conformer the right architecture". The established EEG
deep-learning baselines do, and running them through the *identical* InfoNCE pipeline is what makes the comparison
fair: the objective never sees the frontend, so only the encoder changes.

| arm | frontend | why it is here |
| --- | --- | --- |
| `align_sentence_combined` | `raw_conformer` | the recipe under test |
| `eegnet_clip` | `eegnet` | the compact depthwise-separable standard |
| `deepconvnet_clip` | `deep_conv_net` | the deeper convolutional standard |

Two design notes that are in the code rather than in a footnote:

- **DeepConvNet's canonical four max-pool blocks are tuned for 1000+ samples.** The live `raw_window` is 350 (700 ms
  at 500 Hz) and archived configs use 128. The pooling schedule is adapted to the actual window and raises a clear
  error rather than producing a zero-length time axis.
- **EEGNet's depthwise convolution already *is* a spatial filter over electrodes.** Stacking spherical-harmonic
  spatial encoding in front of it double-counts the geometry, so if both are on the run logs a warning. Treat that
  combination as an explicit ablation, never a default.

In [ ]:
# Same TRAIN_FLAGS as every other arm, `--spatial exact` included. All three configs already set
# `spatial_encoding: spherical_harmonics`, so dropping the flag here would not remove the spatial code -- it would
# only leave the montage CSV unbuilt and silently downgrade these two arms to the approximate cap, making the
# comparison differ by two levers instead of one.
BENCH_ARMS = {
    'benchmark_eegnet_clip': 'experiments/benchmark/eegnet_clip.yaml',
    'benchmark_deepconvnet_clip': 'experiments/benchmark/deepconvnet_clip.yaml',
}

for stem, cfg in BENCH_ARMS.items():
    name = f'{stem}_lo{HOLDOUT}_s{SEED}'
    print(f'\n===== {name} =====')
    !uv run zte-run --config "{cfg}" --root "{DATA_DIR}" --name "{name}" --loso-holdout {HOLDOUT} \
        --out-root "{OUT_ROOT}" {TRAIN_FLAGS} --resume
    mirror_to_drive()

In [ ]:
# A run directory is `<name>` when --name is given, which every training cell above does.
BENCH_RUNS = [f'align_sentence_combined_lo{HOLDOUT}_s{SEED}', *(f'{s}_lo{HOLDOUT}_s{SEED}' for s in BENCH_ARMS)]

# The conformer baseline is BENCH_RUNS[0] and section 10 already audited it into {SUITE}/hardneg. `--out` is part
# of the done signature, so pointing at a new directory would recompute the whole thing; read it where it is.
BENCH_AUDIT = {
    run: f'{SUITE}/hardneg/{run}' if run in HARDNEG_RUNS else f'{SUITE}/benchmark/{run}' for run in BENCH_RUNS
}

for run, out in BENCH_AUDIT.items():
    if run in HARDNEG_RUNS:
        continue
    ckpt = resolve_ckpt(run)
    !uv run zte-rebaseline --ckpt "{ckpt}" --root "{DATA_DIR}" --piece-oracle --out "{out}"

In [ ]:
rows = []
for run in BENCH_RUNS:
    payload = audit('rebaseline', BENCH_AUDIT[run], markdown=False)['payload']
    if not payload:
        rows.append({'arm': run, 'rank pct': None, 'clears': 'NOT SCORED -- rerun its rebaseline cell'})
        continue
    honest = ((payload.get('grid') or {}).get('train_fitted') or {}).get('length_stratified') or {}
    floor = payload.get('floor_comparison') or {}
    hits = honest.get('top1')
    row = {'arm': run, 'frontend': (payload.get('provenance') or {}).get('frontend')}
    row['rank pct'] = num(honest.get('rank_percentile'))
    row['CI'] = [num(v) for v in (honest.get('rank_percentile_ci') or [])[1:]] or None
    row['Top-1 hits'] = None if hits is None else round(hits * honest['n_queries'])
    row['of'] = honest.get('n_queries')
    row['length floor'] = num(floor.get('oracle'))
    row['clears'] = floor.get('clears_floor')
    row['bits from EEG'] = num((payload.get('bit_budget') or {}).get('bits_from_eeg'), 2)
    row['gallery'] = floor.get('gallery')
    row['postproc'] = (payload.get('menu') or {}).get('postprocess_fit')
    rows.append(row)

display(pd.DataFrame(rows))

**The honest framing of this table.** If no arm clears the floor, the benchmark does not say the conformer is better
or worse than EEGNet — it says the readout is confound-bound for all three, and the architecture is not the binding
constraint. That is a more useful statement to the field than a ranking inside the noise, and it is the outcome the
rest of this notebook's evidence predicts.

## 12 · Experiment 7 — where and when the readout comes from

### Why this is occlusion and not attention

The natural ask is "plot the conformer's temporal attention and show it peaks near 400 ms, the N400".
Two different weights hide in that sentence. `RawConformer`'s attentive temporal *pool* exists only under
`conformer_temporal_pool: attention`, which **no live config sets** — every arm pools by a plain mean, so
those weights are not in any trained model. The intra-word transformer's self-attention *is* in every
trained model, and `zte-lens attention` reads it through forward hooks, together with the electrode
mixer's; `notebooks/tbme/zte_attention.ipynb` draws both as the temporal curve and the scalp map.

This section measures **causal contribution** instead: occlude a time span of the raw window, re-embed,
and measure how far the sentence vector moves. That works on every checkpoint and it is the better
instrument regardless — attention weights are famously not explanations, whereas an occlusion drop is a
counterfactual. Read the attention notebook beside this section, never instead of it.

A **null band** travels with the profile: a same-width occlusion at a random offset, so the curve has a
floor rather than being a bare bar chart of drops.

In [ ]:
# One pass produces both halves of this section: `--temporal` the latency profile, `--html` the page carrying the
# channel saliency and its scalp map. Running the two separately would repeat the occlusion work.
LENS_CKPT = resolve_ckpt(f'align_sentence_combined_lo{HOLDOUT}_s{SEED}')
!uv run zte-lens encode --ckpt "{LENS_CKPT}" --root "{DATA_DIR}" --temporal --html \
    --temporal-bins 14 --temporal-sentences 12 --out "{SUITE}/lens"

In [ ]:
show(audit('temporal', f'{SUITE}/lens'))

In [ ]:
TEMPORAL = audit('temporal', f'{SUITE}/lens', markdown=False)['payload']
bins = (TEMPORAL or {}).get('bins') or []

if bins:
    null_band = TEMPORAL.get('null_band') or {}
    fig = go.Figure()
    fig.add_scatter(
        x=[b['center_ms'] for b in bins],
        y=[b['mean_drop'] for b in bins],
        error_y={
            'type': 'data',
            'array': [b['ci_high'] - b['mean_drop'] for b in bins],
            'arrayminus': [b['mean_drop'] - b['ci_low'] for b in bins],
        },
        mode='lines+markers',
        name='occlusion drop',
    )
    if null_band.get('mean_drop') is not None:
        fig.add_hline(
            y=null_band['mean_drop'],
            line_dash='dot',
            line_color='grey',
            annotation_text='random-offset null band',
            annotation_position='top left',
        )
    lo, hi = TEMPORAL.get('n400_window_ms', [300, 500])
    fig.add_vrect(
        x0=lo,
        x1=hi,
        fillcolor='orange',
        opacity=0.12,
        line_width=0,
        annotation_text=f'{lo}-{hi} ms',
        annotation_position='top right',
    )
    fig.update_layout(
        title='Causal contribution by latency within the word window',
        xaxis_title='ms from word onset',
        yaxis_title='cosine drop when occluded',
        height=440,
    )
    fig.show()

    peak = TEMPORAL.get('peak') or {}
    print(f'window            : {TEMPORAL.get("window_ms")} ms over {TEMPORAL.get("raw_window_samples")} samples')
    print(f'readings / words  : {TEMPORAL.get("n_readings")} / {TEMPORAL.get("n_words")}')
    print(f'peak bin          : {peak.get("start_ms")}-{peak.get("end_ms")} ms   above null: {peak.get("above_null")}')
    print(f'peak in {TEMPORAL.get("n400_window_ms")} ms : {TEMPORAL.get("peak_in_n400_window")}')
    print(f'\n{TEMPORAL.get("caveat", "")}')
else:
    print(f'No temporal profile under {SUITE}/lens. Re-run the lens cell above and check it wrote temporal.json;')
    print('the instrument also writes nothing for a band-power checkpoint, though every arm here is raw.')

**The claim this does *not* license.** ZuCo word windows come from eye-tracking segmentation and overlap their
neighbours, so a peak in 300–500 ms is *consistent with* an N400 and is not proof of one. `peak_in_n400_window`
gates nothing, and the profile carries the lens disclaimer: it is an inspection surface, not a result.

### The scalp side, and the one flag that decides whether it means anything

A topographic map of channel importance is only interpretable if the channel axis maps to real electrode positions.

Nearly every config in this repository — the alignment, parallax, decoder and benchmark arms alike — sets
`dataset.montage_csv: res/montage_gsn105.csv`, and `res/` is gitignored. `--spatial exact` is in `TRAIN_FLAGS`, so
every arm this notebook trains gets a real montage: staged from the persistent store, built through `mne`, or copied
from the ZuCo-105 montage shipped inside the package. A run that asked for it and would train on the placeholder cap
now stops with the reason instead, and every run records what it trained on under `electrode_geometry` in its
`manifest.json`.

Two things follow that are easy to get wrong:

- **The mathematics is unaffected by the fallback.** The Fibonacci cap is a genuine set of well-separated points on
  the sphere, so rotation structure and the addition theorem hold exactly for *those* points. What is lost is the
  correspondence to a head: a topoplot from such a run shows which **array indices** mattered, not which regions.
- **A warning at analysis time is not a corrupted analysis, and the flag to read is the checkpoint's own.**
  `harmonics`, `degrees` and `approximate` are *persistent* buffers, so a loaded model carries its trained basis
  whatever this VM holds. `zte-colab geometry --ckpt` reads them directly, and `zte-lens attention` draws a scalp map
  only on coordinates that provably rebuild that basis. `notebooks/tbme/zte_attention.ipynb` §4a runs the check, and
  §4b retrains one fold with the montage present if the basis really is the placeholder.

**The condition for a regional claim is the checkpoint's own `approximate_geometry` flag reading `False` and a
montage that reproduces its basis** — never the YAML, which says nothing about whether the file existed, and never
the absence of a warning.

In [ ]:
LENS = audit('lens', f'{SUITE}/lens', markdown=False)['payload'] or {}
channels = LENS.get('channel_saliency') or {}

if channels:
    top = sorted(zip(channels['labels'], channels['scores']), key=lambda p: -p[1])[:8]
    print(f'electrodes scored : {len(channels["labels"])}   ({channels.get("method")})')
    print(f'regions           : {", ".join(sorted(set(channels.get("regions") or [])))}')
    print('\nlargest cosine drop when occluded:')
    for label, score in top:
        print(f'  {label:<8} {score:.4f}')
    print(f'\npage: {SUITE}/lens/*/LENS.html   (open it from Drive, or download it)')
else:
    print('No channel saliency: this checkpoint carries no montage, so the lens drops the scalp panel')
    print('rather than inventing a layout. Retrain the arm with --spatial exact.')

The lens writes channel saliency into `lens.json` and, with `--html`, a `LENS.html` page carrying the scalp map.
`channel_saliency` is `null` when the checkpoint was trained without a montage, and the page drops the scalp panel
with a note rather than inventing a layout — so an uninterpretable map cannot be produced by accident. Check
`approximate_geometry` before writing a regional sentence about it.

## 13 · Experiment 8 — the full twelve-fold LOSO, with statistics

A single held-out subject is a sample of one. Nothing in this notebook may be quoted as a population result without
this section, and reviewers will not accept it otherwise.

`zte-loso-summary` aggregates the folds. Two things about what it reports:

- The headline is the **rank percentile**, aggregated across folds — not the per-fold pooled Top-1 that appears in a
  session `INDEX.md`, which is the inflated number.
- The spread is a **sample** (n−1) standard deviation. At *n* = 12 the population form under-reports it by about 4%,
  which matters when the spread is the finding.

Section 6b already trains the twelve folds if you ran it. This section reads them.

In [ ]:
# `zte-loso-summary` keys folds on the HOLDOUT ALONE, so pointing it at the whole run root would average one
# level's twelve folds together with every other level, the parallax arms and the benchmark arms. Name the fold
# directories for one level at a time. `--out` is the Markdown path; a `.csv` of the same stem lands beside it.
for level in LEVELS:
    fold_dirs = ' '.join(f'"{OUT_ROOT}/align_{level}_combined_lo{f}_s{SEED}"' for f in FOLDS)
    print(f'\n===== {level} =====')
    !uv run zte-loso-summary --experiments {fold_dirs} --out "{SUITE}/loso/LOSO_{level}.md"

In [ ]:
from IPython.display import Markdown, display

for level in LEVELS:
    summary = pathlib.Path(f'{SUITE}/loso/LOSO_{level}.md')
    if not summary.is_file():
        print(f'no LOSO summary for {level} at {summary}; train its folds in section 6b first.')
        continue

    display(Markdown(f'### {level}'))
    display(Markdown(summary.read_text()))
    display(pd.read_csv(summary.with_suffix('.csv')))

**Read the held-out lift, and read it beside the floor.** A positive lift over chance on the unstratified gallery is
not the same claim as clearing the length oracle — the lift here is referenced to chance, and chance on that gallery
is 1/700.

**This section's numbers do not reach the evidence board.** `zte-evidence` assembles five claim families and LOSO is
not one of them; what the board carries for the granularity levels is the *cross-fold aggregate* from `zte-levels`
in section 6d, which is the same folds read through the floor-aware path. Quote the mean ± sd from the tables above
directly, and say which gallery it was measured on.

## 14 · The evidence board

One command reads every artifact this notebook produced and assembles the claims. It **recomputes nothing** — each
row is read from the audit that wrote it, so the board cannot disagree with the runs it describes.

Three properties make it airtight:

1. **A claim with no floor is `not measured`, never a result.** There is no code path by which an unfloored number
   becomes a headline.
2. **The interval must clear the floor, not the point estimate.** A point estimate above a floor with an interval
   straddling it is the shape every retracted result in this project had.
3. **A missing artifact is named, not dropped.** A silently absent row would read as a claim nobody made — which is
   how a gap becomes an implied pass.

The board's own gates are mutation-tested: break the floor comparison and the test suite goes red.

In [ ]:
# Two roots: the suite holds this notebook's audits, and the sentence arm's own run directory holds the
# `rebaseline/rebaseline.json` that section 6c wrote there -- which is the one carrying `observed_top1`, so the
# resolution-limit row gets a verdict rather than the corpus-only oracle's `not measured`.
FLOOR_RUN = f'{OUT_ROOT}/align_sentence_combined_lo{HOLDOUT}_s{SEED}'

# The run directory comes FIRST. Roots are searched in order, and the suite holds section 6a's corpus-only
# `confound_audit.json`, which carries no `observed_top1` -- reached first, it would shadow the run's own
# `rebaseline/rebaseline.json` and leave the resolution-limit row permanently `not measured`.
!uv run zte-evidence --roots "{FLOOR_RUN}" "{SUITE}" --title "ZTE evidence board - {RUN_DATE}" \
    --out "{SUITE}/board"

In [ ]:
show(audit('evidence', f'{SUITE}/board'))

In [ ]:
BOARD = audit('evidence', f'{SUITE}/board', markdown=False)['payload']

rows = []
for row in BOARD['claims']:
    cell = {'claim': row['key'], 'verdict': row['verdict']}
    cell['value'] = num(row['value'])
    cell['CI'] = None if not row['ci'] else [num(v) for v in row['ci']]
    cell['floor'] = num(row['floor'])
    cell['floor is'] = row['floor_name']
    cell['quotable alone'] = row['headline_safe']
    rows.append(cell)

display(pd.DataFrame(rows))

print(f'\n{BOARD["n_headline_safe"]} of {len(BOARD["claims"])} claim(s) may be quoted without the floor sentence.')
for key, why in sorted(BOARD['missing'].items()):
    print(f'  not measured: {key} - {why}')

### What to do with this

**If a row is green**, it clears a brain-free floor with its interval and may be quoted on its own. Say which floor,
which gallery, and which `postprocess_fit` alongside it.

**If every row is below its floor**, that is the result, and it is publishable: it establishes a *resolution limit*
for non-invasive EEG on this corpus rather than a failure of one model. The field currently reports token- and
word-level decoding numbers on galleries where spelling resolves almost every item; a measured floor that no
architecture clears is the corrective standard, and the piece oracle is the instrument that makes it checkable by
anyone.

**If a row is `not measured`**, run the section that produces it before writing about it.

State the null plainly. On work aimed at people with ALS and locked-in syndrome, an overclaimed result is worse than
no result.

## 15 · Persist, resume, continue

Everything above already lives on Drive. This snapshots the session so a fresh VM — or a fresh month — picks it up
unchanged. Every long cell is resumable and `--resume` is idempotent, so a reclaimed runtime costs only the cell it
was inside.

In [ ]:
mirror_to_drive()
show_resources()

print(f'\nsuite    : {SUITE}')
print(f'board    : {SUITE}/board/evidence.md')
print(f'session  : {DRIVE_DIR}')
print(f'\nTo resume on a new VM: set RESUME_DATE = {RUN_DATE!r} in section 4, then run from section 1.')